# 🚀 Rhushya OpenEnv — Email Triage GRPO Training
**Model:** `Qwen/Qwen2.5-1.5B` via Unsloth FastLanguageModel | **GPU:** T4 Free Tier

## Step 0A · Fix Version Conflicts — Run This First!

In [ ]:
# Kill incompatible packages
!pip uninstall mergekit trl -y -q


## Step 0B · Install Pinned Compatible Stack

In [2]:
!pip install -q \
  "trl==0.8.6" \
  "transformers==4.40.2" \
  "accelerate==0.30.1" \
  "pydantic==2.7.1" \
  "datasets==2.19.1" \
  "huggingface_hub>=0.23.0" \
  "bitsandbytes>=0.43.0"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.0/138.0 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.3/107.3 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.2/245.2 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 93.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.6/302.6 kB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 409.3/409.3 kB 40.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.0/542.0 kB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 89.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 51.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 17.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.0/172.0 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 119.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━

## Step 0C · Install Unsloth (FastLanguageModel — 2x faster on T4)

In [3]:
import subprocess, sys

# Detect CUDA version
result = subprocess.run(['nvcc', '--version'], capture_output=True, text=True)
cuda_ver = "cu121"  # T4 default in Colab
if "12.4" in result.stdout or "12.5" in result.stdout:
    cuda_ver = "cu124"

print(f"Detected CUDA slot: {cuda_ver}")
!pip install -q "unsloth[{cuda_ver}-torch230] @ git+https://github.com/unslothai/unsloth.git"


Detected CUDA slot: cu121
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.2/164.2 MB 6.2 MB/s eta 0:00:0000:0100:01
ERROR: Cannot install unsloth because these package versions have conflicting dependencies.
ERROR: ResolutionImpossible: for help visit https://pip.pypa.io/en/latest/topics/dependency-resolution/#dealing-with-dependency-conflicts


## Step 0D · Verify All Imports

In [4]:
import torch
from trl import GRPOConfig, GRPOTrainer
from datasets import Dataset
from transformers import AutoTokenizer

print("✅ torch:", torch.__version__)
print("✅ CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("✅ GPU:", torch.cuda.get_device_name(0))
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ VRAM: {total:.1f} GB")

try:
    from unsloth import FastLanguageModel, PatchFastRL
    print("✅ Unsloth available")
    UNSLOTH_OK = True
except ImportError:
    print("⚠️  Unsloth not available — will use HF standard loading (slower)")
    UNSLOTH_OK = False


ImportError: cannot import name 'GRPOConfig' from 'trl' (/usr/local/lib/python3.12/dist-packages/trl/__init__.py)

## Step 1 · Clone Repo (Clean — No Nested Clone!)

In [ ]:
import os
os.chdir('/content')

if not os.path.exists('/content/OpenEnv'):
    !git clone https://github.com/Rhushya/OpenEnv.git
else:
    print("Repo already cloned, pulling latest...")
    !cd /content/OpenEnv && git pull origin main

os.chdir('/content/OpenEnv')
print("✅ Working dir:", os.getcwd())

# Verify key files exist
import glob
for f in ['envs/email_triage_env/train_grpo.py',
          'envs/email_triage_env/server/email_triage_environment.py',
          'envs/email_triage_env/models.py']:
    exists = os.path.exists(f)
    print(f"{'✅' if exists else '❌'} {f}")


## Step 2 · Smoke Test — Verify Pipeline (Mandatory!)

In [ ]:
import subprocess, sys

result = subprocess.run(
    [sys.executable, 'envs/email_triage_env/train_grpo.py', '--smoke',
     '--model', 'Qwen/Qwen2.5-1.5B'],
    env={**__import__('os').environ, 'PYTHONPATH': 'src:envs'},
    capture_output=False
)
if result.returncode == 0:
    print("\n✅ SMOKE TEST PASSED — Pipeline is ready for full training!")
else:
    print(f"\n❌ SMOKE TEST FAILED (exit code {result.returncode})")
    print("Check errors above before proceeding.")


## Step 3 · Load Model with FastLanguageModel (Unsloth 4-bit)

In [ ]:
import torch, gc

MODEL_NAME = "Qwen/Qwen2.5-1.5B"
MAX_SEQ_LEN = 512
is_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()

gc.collect()
torch.cuda.empty_cache() if torch.cuda.is_available() else None

if UNSLOTH_OK:
    from unsloth import FastLanguageModel, PatchFastRL
    PatchFastRL("GRPO", FastLanguageModel)

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME,
        max_seq_length=MAX_SEQ_LEN,
        load_in_4bit=True,
        fast_inference=True,
        max_lora_rank=8,
        gpu_memory_utilization=0.6,
        dtype=None,  # auto
    )

    model = FastLanguageModel.get_peft_model(
        model,
        r=8,
        target_modules=["q_proj", "v_proj"],
        lora_alpha=8,
        lora_dropout=0,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=42,
    )
    print("✅ Unsloth 4-bit model + LoRA loaded!")
else:
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16 if is_bf16 else torch.float16,
    )
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto",
    )
    print("✅ HF 4-bit model loaded (Unsloth unavailable)")

if torch.cuda.is_available():
    free = (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_reserved(0)) / 1e9
    print(f"✅ VRAM free after load: {free:.2f} GB")


## Step 4 · Build Dataset (Synthetic Email Prompts)

In [ ]:
from datasets import Dataset

DATASET_SIZE = 64

SYSTEM_MSG = (
    "You are an expert email triage coordinator. "
    "Respond ONLY with the three XML tags below — no explanation, no preamble.\n"
    "Format (copy exactly):\n"
    "<category>CATEGORY</category>\n"
    "<priority>N</priority>\n"
    "<escalate>true|false</escalate>\n"
    "Valid categories: billing, support, spam, urgent, marketing, other\n"
    "Priority: 1 (lowest) to 5 (critical)\n"
    "Output the XML tags immediately as your first tokens."
)

EMAIL_TEMPLATES = [
    "Subject: Invoice overdue\nHi, my invoice #{seed} hasn't been paid for 30 days. Please help.",
    "Subject: Can't login\nI've been locked out of my account since yesterday. Seed {seed}.",
    "Subject: Buy cheap meds online\nClick here for discounts! ref={seed}",
    "Subject: URGENT data breach\nOur production DB is compromised RIGHT NOW. ticket={seed}",
    "Subject: Newsletter signup\nThanks for subscribing to our marketing list. id={seed}",
    "Subject: Refund request\nI'd like a refund for order {seed}. It arrived damaged.",
    "Subject: Password reset\nuser {seed} requested a password reset link.",
    "Subject: System alert\nCPU usage at 99% on server seed={seed}. Immediate attention needed.",
]

prompts = [
    [
        {"role": "system", "content": SYSTEM_MSG},
        {"role": "user",   "content": EMAIL_TEMPLATES[i % len(EMAIL_TEMPLATES)].format(seed=i)},
    ]
    for i in range(DATASET_SIZE)
]

dataset = Dataset.from_dict({"prompt": prompts})
print(f"✅ Dataset ready: {len(dataset)} prompts")
print("\nSample prompt[0]:")
print(prompts[0][1]['content'])


## Step 5 · Define 5 Reward Functions

In [ ]:
import re, sys, os, threading

sys.path.insert(0, 'src')
sys.path.insert(0, 'envs')
sys.path.insert(0, 'envs/email_triage_env')

from server.email_triage_environment import EmailTriageEnvironment
from models import EmailTriageAction

_CACHE: dict = {}
_LOCK = threading.Lock()

def _text(obj):
    if isinstance(obj, str): return obj
    if isinstance(obj, list):
        for item in reversed(obj):
            if isinstance(item, dict) and "content" in item:
                return str(item["content"])
    return str(obj)

def _score(prompt, completion):
    prompt_text = _text(prompt)
    completion_text = _text(completion)
    cache_key = hash(prompt_text[-100:] + completion_text[:200])
    with _LOCK:
        if cache_key in _CACHE:
            return _CACHE[cache_key]

    m = re.search(r"seed[:\s]+(\d+)", prompt_text, re.IGNORECASE)
    seed = int(m.group(1)) if m else 0

    cat_m = re.search(r"<category>(.*?)</category>", completion_text, re.IGNORECASE)
    pri_m = re.search(r"<priority>(\d+)</priority>", completion_text, re.IGNORECASE)
    esc_m = re.search(r"<escalate>(true|false)</escalate>", completion_text, re.IGNORECASE)

    cat = cat_m.group(1).strip().lower() if cat_m else "other"
    pri = max(1, min(5, int(pri_m.group(1)))) if pri_m else 1
    esc = esc_m.group(1).lower() == "true" if esc_m else False
    format_ok = cat_m is not None and pri_m is not None and esc_m is not None
    hacking_penalty = 1.0 if format_ok else -1.0

    try:
        env = EmailTriageEnvironment(difficulty="easy")
        env.reset(seed=seed)
        action = EmailTriageAction(category=cat, priority=pri, should_escalate=esc)
        obs = env.step(action)
        info = obs.info or {}
        comps = info.get("reward_components", {})
        if comps:
            quality   = float(comps.get("quality", 0.0))
            sla       = float(comps.get("sla", 0.0))
            policy    = float(comps.get("policy", 0.0))
            oversight = float(comps.get("oversight", 0.0))
        else:
            cs = float(info.get("category_score", 0.0))
            ps = float(info.get("priority_score", 0.0))
            es = float(info.get("escalation_score", 0.0))
            quality   = 0.5*cs + 0.2*ps + 0.3*es
            sla       = 1.0
            policy    = 1.0
            oversight = float(info.get("task_score", 0.0))
        result = {"quality": quality, "sla": sla, "policy": policy,
                  "oversight": oversight, "hacking": hacking_penalty}
        del env
    except Exception:
        result = {"quality": 0.0, "sla": 0.0, "policy": 0.0,
                  "oversight": 0.0, "hacking": hacking_penalty}
    with _LOCK:
        _CACHE[cache_key] = result
    return result

def reward_quality(prompts, completions, **kw):
    return [_score(p,c)["quality"] for p,c in zip(prompts,completions)]

def reward_sla(prompts, completions, **kw):
    return [_score(p,c)["sla"] for p,c in zip(prompts,completions)]

def reward_policy(prompts, completions, **kw):
    return [_score(p,c)["policy"] for p,c in zip(prompts,completions)]

def reward_oversight(prompts, completions, **kw):
    return [_score(p,c)["oversight"] for p,c in zip(prompts,completions)]

def reward_format(prompts, completions, **kw):
    return [_score(p,c)["hacking"] for p,c in zip(prompts,completions)]

ALL_REWARDS = [reward_quality, reward_sla, reward_policy, reward_oversight, reward_format]
print(f"✅ {len(ALL_REWARDS)} reward functions registered")


## Step 6 · Configure GRPO & Train

In [ ]:
from trl import GRPOConfig, GRPOTrainer
import torch

is_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()

try:
    import bitsandbytes
    optim = "paged_adamw_8bit"
except ImportError:
    optim = "adamw_torch"
print(f"Optimizer: {optim}")

OUTPUT_DIR = "oversight-arena-grpo-qwen25-1.5b"
MAX_STEPS  = 50

grpo_kwargs = dict(
    output_dir=OUTPUT_DIR,
    max_steps=MAX_STEPS,
    learning_rate=5e-6,
    optim=optim,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_generations=4,
    max_completion_length=300,
    temperature=0.9,
    logging_steps=1,
    save_steps=25,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    report_to="none",
    bf16=is_bf16,
    fp16=not is_bf16,
    dataloader_pin_memory=False,
)

try:
    config = GRPOConfig(max_prompt_length=256, **grpo_kwargs)
    print("GRPOConfig: with max_prompt_length")
except TypeError:
    config = GRPOConfig(**grpo_kwargs)
    print("GRPOConfig: without max_prompt_length (TRL>=v1.0)")

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=ALL_REWARDS,
    train_dataset=dataset,
    args=config,
)

print(f"\n{'='*55}")
print(f"  Training: Qwen/Qwen2.5-1.5B (4-bit + LoRA)")
print(f"  Steps: {MAX_STEPS} | Rewards: 5 independent signals")
print(f"  Output: {OUTPUT_DIR}")
print(f"{'='*55}\n")

trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"\n✅ Training complete! Saved to: {OUTPUT_DIR}")


## Step 7 · Quick Inference Test (Sanity Check)

In [ ]:
import torch

TEST_EMAILS = [
    "Subject: Payment overdue\nMy invoice #42 is 45 days unpaid. This is urgent!",
    "Subject: Win a free iPhone!\nClick now to claim your prize. Limited offer!",
    "Subject: Server down\nProduction database is unreachable. All services affected!",
]

if UNSLOTH_OK:
    from unsloth import FastLanguageModel
    FastLanguageModel.for_inference(model)

SYSTEM_MSG_INF = (
    "You are an expert email triage coordinator. "
    "Respond ONLY with:\n"
    "<category>CATEGORY</category>\n"
    "<priority>N</priority>\n"
    "<escalate>true|false</escalate>"
)

print("="*55)
print("INFERENCE TEST — 3 emails")
print("="*55)

for i, email in enumerate(TEST_EMAILS):
    messages = [
        {"role": "system", "content": SYSTEM_MSG_INF},
        {"role": "user",   "content": email},
    ]
    input_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(input_text, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=80,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    response = tokenizer.decode(
        outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True
    )
    print(f"\n[Email {i+1}] {email[:50]}...")
    print(f"[Response] {response.strip()}")

print("\n✅ Inference test complete!")


## Step 8 · Push to Hugging Face Hub

In [ ]:
HUB_REPO = "Rhushya/oversight-arena-qwen25-1.5b"

!huggingface-cli login --token YOUR_HF_WRITE_TOKEN_HERE


In [ ]:
from huggingface_hub import HfApi

api = HfApi()
api.upload_folder(
    folder_path=OUTPUT_DIR,
    repo_id=HUB_REPO,
    repo_type="model",
    commit_message="GRPO-trained email triage model — Qwen2.5-1.5B 4-bit LoRA",
)
print(f"\n✅ Model live at: https://huggingface.co/{HUB_REPO}")


## Step 9 · Update Space README with Model Link

In [ ]:
readme_text = f"""---
title: Oversight Inbox Arena
emoji: 🛡️
colorFrom: orange
colorTo: red
sdk: docker
pinned: true
---

# Oversight Inbox Arena

Multi-Agent RL Email Triage — Grand Finale Demo

**Trained Model:** https://huggingface.co/{HUB_REPO}
"""

with open("SPACE_README.md", "w") as f:
    f.write(readme_text)

print(readme_text)
print("\n✅ Update your HF Space README with the model link above.")
print(f"Space: https://huggingface.co/spaces/Rhushya/email-triage-env-openenv")


## Step 10 · Push Notebook to GitHub

In [ ]:
!git config user.email "rhushyakc@gmail.com"
!git config user.name "Rhushya KC"

# Download this notebook from Colab first, then:
# !cp /path/to/Rhushya_OpenEnv_EmailTriage_Training.ipynb .
# !git add Rhushya_OpenEnv_EmailTriage_Training.ipynb
# !git add envs/email_triage_env/train_grpo.py
# !git commit -m "Final training notebook + Qwen2.5-1.5B GRPO"
# !git push https://<GITHUB_TOKEN>@github.com/Rhushya/OpenEnv.git main

print("📋 Copy-paste the commands above with your GitHub token to push.")
print(f"\n🏁 FINAL SUBMISSION LINKS:")
print(f"   Space:  https://huggingface.co/spaces/Rhushya/email-triage-env-openenv")
print(f"   Model:  https://huggingface.co/{HUB_REPO}")
print(f"   Repo:   https://github.com/Rhushya/OpenEnv")
